# 24 — Ablation v2: sampler, encoder and LDM diagnostics, and why Flow-Matching makes fewer boundary errors

Follow-up to notebook 18 (2026-11-03/04). Everything here uses the **existing ablation_v2 checkpoints**; nothing is retrained.
Metric definitions, QSS frames (last 5 test frames), ensemble size (5) and colour scales follow notebook 18.

| Data (W&B, single-version artifacts) | Produced by |
|---|---|
| `ablation_v2_preview` (v3: DiffusionSR/LDM sampled with **DDPM-1000**, FM with Euler-50) | `scripts/precompute_ablation_v2_preview.py` |
| `ablation_v2_sampler_diagnostics` (DDIM 50/200/1000, DDPM-1000, Euler 50/200, encoder alone, LDM VAE checks) | `scripts/diag_ablation_v2_sampling.py` |
| `ablation_v2_fm_vs_ddpm_probes` (probe A: one-step denoising; probe B: sampling trajectories) | `scripts/probe_fm_vs_ddpm.py` |

**Findings**
1. **DiffusionSR's stray pixels come from deterministic DDIM sampling, not from too few steps.** DDIM 50, 200 and 1000 steps give the same result; ancestral **DDPM** removes the strays and lowers MP-MAE. DDPM is now the default sampler for DiffusionSR/LDM.
2. **The frozen RRDB encoder alone scores about as well as the diffusion models on PSNR/MP-MAE**, but is too smooth; the diffusion stage mainly restores sharp boundaries/texture.
3. **LDM's failure is its latent U-Net, not the VAE**: the VAE reconstructs GT (and the encoder output) at ~29 dB, LDM samples reach only ~19 dB with every sampler; its latent U-Net has ~0.07 M parameters vs 13 M in pixel space.
4. **Flow-Matching makes fewer boundary mistakes because it predicts velocity, not noise.** At low noise both networks are equally accurate; at high noise DiffusionSR's implied clean-field estimate degrades sharply (noise-prediction errors are amplified by σ/α), which seeds stray pixels that DDIM then locks in.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
import os, sys, json, tempfile, warnings
from pathlib import Path

WANDB_PROJECT = 'Flow3D_SuperResolution'
WANDB_ENTITY  = os.getenv('WANDB_ENTITY', 'ngng-')

# On TRACE the data is read from the run directories; anywhere else it is downloaded from W&B.
_TRACE_RUNS = Path('/trace/group/forgelab/ngng/multifield/DiffusionSR_shohom/diffusionsr/runs/ablation_v2')
LOCAL_DIRS = {
    'ablation_v2_sampler_diagnostics': _TRACE_RUNS / '_diagnostics',
    'ablation_v2_fm_vs_ddpm_probes':   _TRACE_RUNS / '_diagnostics_fm_vs_ddpm',
}

REPO = Path.cwd().parents[1] if Path.cwd().name == 'notebooks' else Path.cwd()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

MODELS = ['diffusion', 'fm', 'ldm']
ENCS   = ['enc', 'noenc']
FIELDS = ['temp', 'multi']
NORMS  = ['std', 'gstd']
CONFIGS = [f'{f}_{n}' for f in FIELDS for n in NORMS]
MODEL_LABEL = {'diffusion': 'DiffusionSR', 'fm': 'Flow-Matching', 'ldm': 'LDM'}
CFG_LABEL = {'temp_std': 'temperature, pixel-wise std', 'temp_gstd': 'temperature, global std',
             'multi_std': 'multifield, pixel-wise std', 'multi_gstd': 'multifield, global std'}
FRAME = 0            # QSS frame shown in the figures (notebook 18 default: frame 1/5)
FIG_CONFIG = 'multi_std'  # config used for the single-config figures

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt
import wandb
from IPython.display import display, Markdown
from scipy.ndimage import laplace, binary_dilation
from skimage.metrics import structural_similarity as _ssim_fn
from diffusionsr.analysis.analysis_functions import get_profile

warnings.filterwarnings('ignore')
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 30)
print('REPO:', REPO)

In [ ]:
# ── Load the three artifacts (fully into memory) ─────────────────────────────
def _load_dir(name, loader):
    local = LOCAL_DIRS.get(name)
    if local is not None and local.exists():
        print(f'{name}: reading {local}')
        return loader(local)
    art = wandb.Api().artifact(f'{WANDB_ENTITY}/{WANDB_PROJECT}/{name}:latest')
    print(f'{name}: downloading {art.version} from W&B …')
    with tempfile.TemporaryDirectory() as td:
        art.download(root=td)
        return loader(Path(td))

def _npz(p):
    with np.load(str(p)) as z:
        return {k: np.array(z[k]) for k in z.files}

def _load_preview(root):
    out = {}
    for p in root.glob('*/qss_samples.npz'):
        out[tuple(p.parent.name.split('_'))] = _npz(p)
    return out

def _load_diag(root):
    out, stats = {}, {}
    for p in root.glob('*.npz'):
        var, setting = p.stem.split('__')
        out[(var, setting)] = _npz(p)
    for p in root.glob('*__latent_stats.json'):
        stats[p.name.split('__')[0]] = json.loads(p.read_text())
    return out, stats

def _load_probe(root):
    return {p.stem: _npz(p) for p in root.glob('*.npz')}

PREVIEW = _load_dir('ablation_v2_preview', _load_preview)
DIAG, LATENT = _load_dir('ablation_v2_sampler_diagnostics', _load_diag)
PROBE = _load_dir('ablation_v2_fm_vs_ddpm_probes', _load_probe)
print(f'{len(PREVIEW)} preview variants, {len(DIAG)} diagnostic sample sets, {len(PROBE)} probe files')

In [ ]:
# ── Metrics (notebook 18 definitions) and extra diagnostics ─────────────────
BAD_K, MELT_K = 800, 1900   # "badly wrong" pixel threshold; melt-pool threshold used by get_profile

def edge_mask(g):
    """Melt-pool edge band of a GT temperature field: |Laplacian| > 300 K, dilated by 2 px."""
    return binary_dilation(np.abs(laplace(g)) > 300, iterations=2)

def roughness(img):
    """Mean |discrete Laplacian| [K] = T(i±1,j) + T(i,j±1) − 4·T(i,j); 0 in flat regions, large across sharp jumps."""
    return np.abs(laplace(img.astype(np.float64))).mean()

def nb18_metrics(P, G):
    """P, G: (F, H, W) prediction (ensemble mean) and GT temperature. PSNR pooled over frames, peak = max GT."""
    n = P.shape[0]
    mse = np.sum((G - P) ** 2) / (n * G.shape[-2] * G.shape[-1])
    psnr = 20 * np.log10(G.max() / (np.sqrt(mse) + 1e-12))
    dr = float(G.max() - G.min())
    ssim = np.mean([_ssim_fn(P[i], G[i], data_range=dr) for i in range(n)])
    mp = np.mean([np.abs(get_profile(P[i][None])[0] - get_profile(G[i][None])[0]).mean() for i in range(n)])
    return psnr, ssim, mp

def full_metrics(S, G):
    """S: (F, N, H, W) samples, G: (F, H, W). Notebook-18 metrics on the ensemble mean, plus per-sample diagnostics."""
    S = S.astype(np.float64); G = G.astype(np.float64)
    psnr, ssim, mp = nb18_metrics(S.mean(1), G)
    em = np.stack([edge_mask(g) for g in G])
    return {'PSNR': psnr, 'SSIM': ssim, 'MP-MAE': mp,
            'PSNR 1 sample': np.mean([nb18_metrics(S[:, j], G)[0] for j in range(S.shape[1])]),
            'stray px/sample': ((np.abs(S - G[:, None]) > BAD_K) & ~em[:, None]).sum((2, 3)).mean(),
            'spread σ [K]': S.std(1).mean() if S.shape[1] > 1 else np.nan,
            'roughness [K]': np.mean([roughness(im) for im in S.reshape(-1, *S.shape[-2:])])}

# Colour scales: notebook 18 rule (GT p1–p99; error p99 over all preview variants)
_g = np.concatenate([v['gts'][:, 0].ravel() for v in PREVIEW.values()])
VMIN_T, VMAX_T = np.percentile(_g, 1), np.percentile(_g, 99)
VMAX_ERR = np.percentile(np.concatenate([np.abs(v['preds'][:, 0] - v['gts'][:, 0]).ravel() for v in PREVIEW.values()]), 99)
KW = dict(cmap='jet', vmin=VMIN_T, vmax=VMAX_T, origin='lower', aspect='auto')
EKW = dict(cmap='hot', vmin=0, vmax=VMAX_ERR, origin='lower', aspect='auto')
GT_ROUGH = np.mean([roughness(g) for v in [PREVIEW[('diffusion', 'enc', 'temp', 'std')]] for g in v['gts'][:, 0]])
print(f'Temperature scale {VMIN_T:.0f}–{VMAX_T:.0f} K, error scale 0–{VMAX_ERR:.0f} K, GT roughness {GT_ROUGH:.0f} K')

def _ax_img(ax, img, title, kw=KW, fs=9):
    ax.imshow(np.asarray(img, np.float64).T, **kw); ax.set_title(title, fontsize=fs); ax.set_xticks([]); ax.set_yticks([])

## 1. Regenerated preview — what notebooks 18 and 23 now show

`ablation_v2_preview` v3: DiffusionSR and LDM are sampled with ancestral **DDPM over all 1000 steps**, Flow-Matching with its
trained **Euler-50** sampler. Metrics are computed on the mean of 5 samples over the 5 QSS frames, temperature channel
(PSNR/SSIM higher is better; MP-MAE = mean melt-pool depth error in pixels, lower is better).

In [ ]:
_b = PREVIEW[('diffusion', 'enc', 'temp', 'std')]
bp, bs, bm = nb18_metrics(_b['ulrs'][:, 0], _b['gts'][:, 0])
print(f'Bicubic-upscaled LR input (baseline): PSNR {bp:.2f} dB | SSIM {bs:.3f} | MP-MAE {bm:.2f} px\n')
rows = []
for (m, e, f, n), v in PREVIEW.items():
    p, s, mp = nb18_metrics(v['preds'][:, 0], v['gts'][:, 0])
    rows.append({'Model': MODEL_LABEL[m], 'Conditioning': {'enc': 'encoder', 'noenc': 'none (unconditional)'}[e],
                 'Config': f'{f}_{n}', 'Sampler': f"{v['sampling_sampler']}-{int(v['sampling_n_sampling_steps'])}",
                 'PSNR': p, 'SSIM': s, 'MP-MAE': mp})
prev_df = pd.DataFrame(rows)
for col in ['PSNR', 'SSIM', 'MP-MAE']:
    t = prev_df.pivot_table(index=['Conditioning', 'Model'], columns='Config', values=col)[CONFIGS]
    display(Markdown(f'**{col}**'))
    display(t.style.format('{:.3f}' if col == 'SSIM' else '{:.2f}')
             .background_gradient(cmap='RdYlGn_r' if col == 'MP-MAE' else 'RdYlGn', axis=None))

In [ ]:
# Overlays: encoder runs, all four configs (prediction = mean of 5 samples; bottom row = |error|)
for cfg in CONFIGS:
    f, n = cfg.split('_')
    fig, ax = plt.subplots(2, 5, figsize=(18, 6.4), gridspec_kw={'hspace': 0.3, 'wspace': 0.08})
    G = PREVIEW[('diffusion', 'enc', f, n)]['gts'][FRAME, 0]
    _ax_img(ax[0, 0], G, 'Ground Truth'); _ax_img(ax[1, 0], PREVIEW[('diffusion', 'enc', f, n)]['ulrs'][FRAME, 0], 'Upscaled LR (input)')
    for c, m in enumerate(MODELS, 1):
        v = PREVIEW[(m, 'enc', f, n)]
        r = prev_df[(prev_df.Model == MODEL_LABEL[m]) & (prev_df.Config == cfg) & (prev_df.Conditioning == 'encoder')].iloc[0]
        _ax_img(ax[0, c], v['preds'][FRAME, 0], f"{MODEL_LABEL[m]} ({r.Sampler})\nPSNR {r.PSNR:.1f} | SSIM {r.SSIM:.3f} | MP-MAE {r['MP-MAE']:.2f}")
        _ax_img(ax[1, c], np.abs(v['preds'][FRAME, 0] - G), f'|error| {MODEL_LABEL[m]}', EKW)
    # encoder alone (from the diagnostics) for reference
    eo = DIAG[(f'diffusion_enc_{cfg}', 'enc_only')]['samples'][FRAME, 0, 0]
    _ax_img(ax[0, 4], eo, 'RRDB encoder alone (no diffusion)'); _ax_img(ax[1, 4], np.abs(eo - G), '|error| encoder alone', EKW)
    fig.suptitle(f'Encoder runs | {CFG_LABEL[cfg]} | QSS frame {FRAME + 1}/5', fontsize=12)
    plt.show()

## 2. Does the sampler matter?

Each **encoder run** was re-sampled on the same 5 QSS frames × 5 samples. All DDIM settings start from the **same initial
noise**, so any difference between them comes from the step grid alone.

* **DiffusionSR**: DDIM (η = 0, deterministic) with 50 / 200 / 1000 steps, and ancestral DDPM with 1000 steps (fresh noise every step).
* **Flow-Matching**: Euler with 50 (trained setting) and 200 steps.

Extra columns:
* **PSNR 1 sample** — PSNR of a single draw (the ensemble mean hides part of the per-sample error).
* **stray px/sample** — pixels more than 800 K wrong that lie **away from the melt-pool edge** (GT edge band = |Laplacian| > 300 K, dilated 2 px). These are the isolated "blobs".
* **spread σ** — mean per-pixel standard deviation across the 5 samples.
* **roughness** — mean absolute discrete Laplacian, |T(i±1,j) + T(i,j±1) − 4·T(i,j)|, in K, averaged over pixels, frames and **individual samples**. Zero in flat regions, large across sharp jumps, so it is dominated by how sharp the melt-pool boundary is. It also rises with speckle, so **closer to the GT value is better, not higher**.

In [ ]:
rows = []
for (var, setting), d in DIAG.items():
    m, _, f, n = var.split('_')
    if setting in ('vae_recon_gt', 'vae_recon_enc'):
        continue
    rows.append({'Model': MODEL_LABEL[m], 'Config': f'{f}_{n}', 'Setting': setting,
                 's / frame (5 samples)': float(d['seconds']) / d['samples'].shape[0],
                 **full_metrics(d['samples'][:, :, 0], d['gts'][:, 0])})
diag_df = pd.DataFrame(rows)
ORDER = ['enc_only', 'ddim50', 'ddim200', 'ddim1000', 'ddpm1000', 'euler50', 'euler200']
diag_df['Setting'] = pd.Categorical(diag_df.Setting, ORDER, ordered=True)
cols = ['PSNR', 'SSIM', 'MP-MAE', 'PSNR 1 sample', 'stray px/sample', 'spread σ [K]', 'roughness [K]', 's / frame (5 samples)']
print(f'GT roughness: {GT_ROUGH:.0f} K\n')
display(Markdown('**Mean over the 4 field/normalisation configs** (DiffusionSR and Flow-Matching; LDM is in section 4)'))
display(diag_df[diag_df.Model != 'LDM'].groupby(['Model', 'Setting'], observed=True)[cols].mean()
        .style.format('{:.3f}', subset=['SSIM']).format('{:.2f}', subset=[c for c in cols if c != 'SSIM']))
display(Markdown('**Per config**'))
display(diag_df[diag_df.Model != 'LDM'].set_index(['Model', 'Config', 'Setting']).sort_index()[cols].round(3))

In [ ]:
# Sampler sweep figures: mean of 5 / one sample / |error| of that sample
SWEEP = [('diffusion', 'ddim50', 'DiffusionSR DDIM-50'), ('diffusion', 'ddim200', 'DiffusionSR DDIM-200'),
         ('diffusion', 'ddim1000', 'DiffusionSR DDIM-1000'), ('diffusion', 'ddpm1000', 'DiffusionSR DDPM-1000'),
         ('fm', 'euler50', 'Flow-Matching Euler-50'), ('fm', 'euler200', 'Flow-Matching Euler-200')]
for cfg in CONFIGS:
    fig, ax = plt.subplots(3, len(SWEEP) + 1, figsize=(3.0 * (len(SWEEP) + 1), 8.4))
    G = DIAG[(f'diffusion_enc_{cfg}', 'ddim50')]['gts'][FRAME, 0]
    _ax_img(ax[0, 0], G, 'Ground Truth'); _ax_img(ax[1, 0], DIAG[(f'diffusion_enc_{cfg}', 'ddim50')]['ulrs'][FRAME, 0], 'Upscaled LR'); ax[2, 0].axis('off')
    for c, (m, s, lbl) in enumerate(SWEEP, 1):
        S = DIAG[(f'{m}_enc_{cfg}', s)]['samples'][FRAME, :, 0]
        r = diag_df[(diag_df.Model == MODEL_LABEL[m]) & (diag_df.Config == cfg) & (diag_df.Setting == s)].iloc[0]
        _ax_img(ax[0, c], S.mean(0), f"{lbl}\nmean of 5: PSNR {r.PSNR:.1f} | MP-MAE {r['MP-MAE']:.2f}", fs=8)
        _ax_img(ax[1, c], S[0], f"one sample ({r['stray px/sample']:.2f} strays/sample)", fs=8)
        _ax_img(ax[2, c], np.abs(S[0] - G), '|error| of that sample', EKW, fs=8)
    fig.suptitle(f'Sampler sweep, encoder runs | {CFG_LABEL[cfg]} | QSS frame {FRAME + 1} | same initial noise for all DiffusionSR settings', fontsize=11)
    plt.tight_layout(); plt.show()

**Reading the sweep**

* DDIM 50, 200 and 1000 steps are practically identical, with the same stray dots in the same places. More steps only follow
  the same deterministic path more precisely, so **discretisation is not the cause**.
* Ancestral DDPM removes almost all stray pixels and lowers MP-MAE. It costs about 18× more time than DDIM-50 (column
  `s / frame`). **Since 2026-11-03 DDPM is the default sampler for DiffusionSR/LDM** (`precompute_ablation_v2_preview.py`, notebook 23).
* Flow-Matching is unchanged between 50 and 200 Euler steps.

**Why DDIM keeps its mistakes.** At each step both samplers ask the network for its current guess of the clean field x̂₀.
DDIM then builds the next state entirely from the network's own beliefs, x_next = √ᾱ_next·x̂₀ + √(1−ᾱ_next)·ε̂.
If the network wrongly decides a pixel near the keyhole is hot, the next state is constructed to be exactly consistent with
that mistake, and the network agrees with itself from then on. DDPM instead adds **fresh random noise** at every step,
x_next = (mix of x_t and x̂₀) + σ_t·z. That noise partly erases yesterday's decision, so the network has to re-decide from
the conditioning and the neighbourhood; mistakes get washed out over 1000 steps. Section 5 measures this directly.

## 3. Encoder alone vs the diffusion stage

`enc_only` is the frozen RRDB encoder's own pixel output (no diffusion). Each run directory has its own copy of the encoder,
so there are three encoder rows per config.

In [ ]:
best = {'DiffusionSR': 'ddpm1000', 'Flow-Matching': 'euler50', 'LDM': 'ddpm1000'}
t = diag_df[diag_df.apply(lambda r: r.Setting in ('enc_only', best[r.Model]), axis=1)]
display(t.groupby(['Model', 'Setting'], observed=True)[['PSNR', 'SSIM', 'MP-MAE', 'roughness [K]']].mean().round(3))
print(f'GT roughness: {GT_ROUGH:.0f} K')

On PSNR and MP-MAE the encoder alone is as good as DiffusionSR (DDPM) and within ~0.2 dB of Flow-Matching. What the
diffusion stage adds is **sharpness**: the encoder output is too smooth (roughness well below GT), whereas the diffusion
outputs are close to GT's roughness. PSNR and MP-MAE do not reward that, so **report the encoder-alone baseline next to
the diffusion models** (and next to the `lrcond` arm in notebook 23).

## 4. LDM: is it the VAE, the sampler or the latent U-Net?

* `vae_recon_gt` — decode(encode(GT)): the best any model working in this latent space could do.
* `vae_recon_enc` — decode(encode(encoder output)): what LDM's conditioning carries, back in pixel space.
* LDM samples with DDIM-50, DDIM-1000 and DDPM-1000.

In [ ]:
rows = []
for (var, setting), d in DIAG.items():
    if not var.startswith('ldm_'):
        continue
    rows.append({'Config': var[len('ldm_enc_'):], 'Setting': setting, **full_metrics(d['samples'][:, :, 0], d['gts'][:, 0])})
ldm_df = pd.DataFrame(rows)
LORDER = ['vae_recon_gt', 'vae_recon_enc', 'enc_only', 'ddim50', 'ddim1000', 'ddpm1000']
ldm_df['Setting'] = pd.Categorical(ldm_df.Setting, LORDER, ordered=True)
display(Markdown('**Mean over the 4 configs**'))
display(ldm_df.groupby('Setting', observed=True)[['PSNR', 'SSIM', 'MP-MAE', 'stray px/sample', 'spread σ [K]', 'roughness [K]']].mean().round(3))
display(Markdown('**Latent statistics (200 train frames) and model sizes**'))
display(pd.DataFrame({k: {'latent shape': tuple(v['latent_shape']), 'per-channel mean': np.round(v['per_channel_mean'], 2).tolist(),
                          'per-channel std': np.round(v['per_channel_std'], 2).tolist(), 'max |z|': round(v['abs_max'], 1),
                          'latent U-Net params': f"{v['unet_params']:,}", 'VAE params': f"{v['vae_params']:,}"}
                      for k, v in sorted(LATENT.items())}).T)
print('Pixel-space U-Net (DiffusionSR / Flow-Matching): ~13.07 M parameters (dim 60, dim_mults (1, 2, 4))')

In [ ]:
fig, ax = plt.subplots(len(CONFIGS), 7, figsize=(21, 3.0 * len(CONFIGS)))
for r_, cfg in enumerate(CONFIGS):
    var = f'ldm_enc_{cfg}'
    G = DIAG[(var, 'vae_recon_gt')]['gts'][FRAME, 0]
    items = [(G, 'Ground Truth')]
    for s, l in [('vae_recon_gt', 'VAE decode(encode(GT))'), ('enc_only', 'RRDB encoder alone'),
                 ('vae_recon_enc', 'VAE decode(encode(encoder))'), ('ddim50', 'LDM DDIM-50'),
                 ('ddim1000', 'LDM DDIM-1000'), ('ddpm1000', 'LDM DDPM-1000')]:
        rr = ldm_df[(ldm_df.Config == cfg) & (ldm_df.Setting == s)].iloc[0]
        items.append((DIAG[(var, s)]['samples'][FRAME, :, 0].mean(0), f"{l}\nPSNR {rr.PSNR:.1f} | MP-MAE {rr['MP-MAE']:.2f}"))
    for c, (img, ttl) in enumerate(items):
        _ax_img(ax[r_, c], img, ttl, fs=8)
    ax[r_, 0].set_ylabel(CFG_LABEL[cfg], fontsize=9)
fig.suptitle(f'LDM decomposition, QSS frame {FRAME + 1}: what the VAE can represent vs what the latent diffusion produces', fontsize=12)
plt.tight_layout(); plt.show()

The VAE is **not** the bottleneck: it reconstructs GT at ~29.5 dB, and LDM's own conditioning decodes to a ~29 dB answer.
Yet LDM samples score ~19 dB with every sampler, so the failure is the **latent U-Net** (~0.07 M parameters, one resolution
level, built in `LDMModel._rebuild_model_for_latent`), which does not make use of the conditioning it is given. The latents
are close to unit scale (std ≈ 1.4), so latent scaling is not the main issue. A fair "latent vs pixel" comparison would need
a latent U-Net of comparable size.

## 5. Why does Flow-Matching make fewer boundary mistakes?

Both models use the same U-Net, encoder, data, epochs and Huber loss. They differ in **what the network predicts**, and
therefore in how its output is turned back into a guess of the clean field x̂₀:

| | DiffusionSR (predicts the noise ε) | Flow-Matching (predicts the velocity v = ε − x₀) |
|---|---|---|
| noisy input | x_t = α·x₀ + σ·ε | x_t = (1−t)·x₀ + t·ε |
| clean-field guess | x̂₀ = (x_t − σ·ε̂) / α | x̂₀ = x_t − t·v̂ |
| error in the guess | x̂₀ − x₀ = **−(σ/α)**·(ε̂ − ε) | x̂₀ − x₀ = **−t**·(v̂ − v), t ≤ 1 |
| loss weight on clean-field error | (α/σ)²: 1 at σ/α = 1, **1/900 at σ/α = 30** | ((1+σ/α)/(σ/α))²: 4 at σ/α = 1, **≈1.07 at σ/α = 30** |

At high noise (sampling starts at σ/α ≈ 130), x_t is almost pure noise, so the noise target can be predicted almost
perfectly by copying the input. The melt pool is only the faint α·x₀ part, and any error in ε̂ is multiplied by σ/α
when x̂₀ is formed. The velocity target contains x₀ at full strength at every noise level, so Flow-Matching is forced to
output a good clean-field guess even at high noise; its loss penalises a wrong guess there ~1000× more.

Two probes test this on the trained checkpoints (σ/α = noise-to-signal ratio):

* **Probe A — the network alone.** Take the GT frame, add the **same** noise at the **same** σ/α for both models, ask
  each network once for x̂₀. No sampler involved. 4 configs × 5 frames × 4 noise draws.
* **Probe B — along the sampling path.** Run DDIM-50, DDPM-1000 and Flow-Matching Euler-50 from the **same initial
  noise** and record x̂₀ at every step (DDPM: every 20th). For every stray pixel in a final sample, find the step from
  which x̂₀ was wrong (> 800 K) and stayed wrong. 4 configs × 5 frames × 5 samples.

In [ ]:
def _stats(X, G):
    X = X.astype(np.float32); e = np.abs(X - G); em = edge_mask(G)
    return {'mae': e.mean((-2, -1)), 'mae_edge': e[..., em].mean(-1),
            'stray': ((e > BAD_K) & ~em).sum((-2, -1)), 'flips': ((X > MELT_K) != (G > MELT_K)).sum((-2, -1))}

A = {}
for cfg in CONFIGS:
    z = PROBE[f'{cfg}__probeA']; R_A = z['r']
    for m in ['dm', 'fm']:
        acc = {k: [] for k in ['mae', 'mae_edge', 'stray', 'flips']}
        for k in range(z['gts'].shape[0]):
            s = _stats(z[m][k], z['gts'][k])
            for kk in acc: acc[kk].append(s[kk])
        A[(cfg, m)] = {kk: np.concatenate(v).mean(0) for kk, v in acc.items()}
probeA = pd.DataFrame({
    'σ/α': R_A,
    'mean |err| DSR [K]': np.mean([A[(c, 'dm')]['mae'] for c in CONFIGS], 0),
    'mean |err| FM [K]':  np.mean([A[(c, 'fm')]['mae'] for c in CONFIGS], 0),
    'edge-band |err| DSR': np.mean([A[(c, 'dm')]['mae_edge'] for c in CONFIGS], 0),
    'edge-band |err| FM':  np.mean([A[(c, 'fm')]['mae_edge'] for c in CONFIGS], 0),
    'stray px DSR': np.mean([A[(c, 'dm')]['stray'] for c in CONFIGS], 0),
    'stray px FM':  np.mean([A[(c, 'fm')]['stray'] for c in CONFIGS], 0),
    'wrong side of 1900 K DSR': np.mean([A[(c, 'dm')]['flips'] for c in CONFIGS], 0),
    'wrong side of 1900 K FM':  np.mean([A[(c, 'fm')]['flips'] for c in CONFIGS], 0)})
display(Markdown('**Probe A: one-step clean-field guess from GT + the same noise (mean over configs, frames, draws; DSR = DiffusionSR)**'))
display(probeA.style.format('{:.3f}', subset=['σ/α']).format('{:.2f}', subset=probeA.columns[1:]))

fig, ax = plt.subplots(1, 3, figsize=(17, 4.6))
for j, (k, lab) in enumerate([('mae', 'mean |x̂₀ − GT| [K]'), ('mae_edge', 'mean |x̂₀ − GT| in the melt-pool edge band [K]'),
                              ('flips', 'pixels on the wrong side of 1900 K (per frame)')]):
    for m, col, lbl in [('dm', 'tab:blue', 'DiffusionSR (predicts noise)'), ('fm', 'tab:orange', 'Flow-Matching (predicts velocity)')]:
        for c in CONFIGS:
            ax[j].plot(R_A, A[(c, m)][k], color=col, alpha=0.3, lw=1)
        ax[j].plot(R_A, np.mean([A[(c, m)][k] for c in CONFIGS], 0), color=col, lw=2.5, label=lbl)
    ax[j].set_xscale('log'); ax[j].set_yscale('log'); ax[j].set_xlabel('noise-to-signal ratio σ/α'); ax[j].set_title(lab, fontsize=10); ax[j].grid(alpha=0.3)
ax[0].legend(fontsize=9)
fig.suptitle('Probe A: each network\'s one-step guess of the clean field (thin lines: 4 configs; thick: mean)', fontsize=11)
plt.tight_layout(); plt.show()

At low noise (σ/α ≤ 1) DiffusionSR's network is **as good as or slightly better than** Flow-Matching's, so its fine
boundary detail is not the weak point. Above σ/α ≈ 3 its clean-field guess degrades rapidly while Flow-Matching's levels
off: by σ/α = 30–100 DiffusionSR produces tens to hundreds of badly wrong pixels away from the edge, Flow-Matching almost none.

In [ ]:
SAMPLERS = [('ddim', 'tab:blue', 'DiffusionSR DDIM-50 (deterministic)'), ('ddpm', 'tab:green', 'DiffusionSR DDPM-1000 (stochastic)'),
            ('fm', 'tab:orange', 'Flow-Matching Euler-50 (deterministic)')]
B = {s: {'r': None, 'wrong': [], 'persist': [], 'commit_r': [], 'final': []} for s, _, _ in SAMPLERS}
for cfg in CONFIGS:
    z = PROBE[f'{cfg}__probeB']; G = z['gts']
    for s in B:
        T = z[f'traj_{s}'].astype(np.float32); Fn = z[f'final_{s}'].astype(np.float32); r = z[f'r_{s}']; B[s]['r'] = r
        for k in range(G.shape[0]):
            em = edge_mask(G[k])
            off = (np.abs(T[k] - G[k]) > BAD_K) & ~em            # (steps, samples, H, W)
            fin = (np.abs(Fn[k] - G[k]) > BAD_K) & ~em            # (samples, H, W)
            B[s]['wrong'].append(off.sum((-2, -1)).mean(-1))
            w = off.sum((-2, -1)).sum(-1); still = (off & fin[None]).sum((-2, -1)).sum(-1)
            B[s]['persist'].append(np.where(w > 0, still / np.maximum(w, 1), np.nan))
            for nn in range(fin.shape[0]):
                for yy, xx in zip(*np.where(fin[nn])):
                    ok = np.where(~off[:, nn, yy, xx])[0]
                    B[s]['commit_r'].append(r[min(0 if len(ok) == 0 else ok[-1] + 1, len(r) - 1)])
            B[s]['final'].append(fin.sum((-2, -1)).mean())

def _at(s, target):
    r = B[s]['r']; i = int(np.argmin(np.abs(np.log(np.where(np.isfinite(r), r, 1e9)) - np.log(target))))
    return i, r[i]
rows = []
for s, _, lbl in SAMPLERS:
    wrong = np.nanmean(np.stack(B[s]['wrong']), 0); per = np.nanmean(np.stack(B[s]['persist']), 0); cr = np.array(B[s]['commit_r'])
    row = {'Sampler': lbl, 'final stray px/sample': np.mean(B[s]['final']), 'strays in total': len(cr),
           'median σ/α where strays were seeded': np.median(cr) if len(cr) else np.nan}
    for tg in [10, 3, 1, 0.3]:
        i, rv = _at(s, tg)
        row[f'wrong px in x̂₀ @σ/α≈{tg}'] = wrong[i]
        row[f'% still wrong at end @σ/α≈{tg}'] = 100 * np.nan_to_num(per[i])
    rows.append(row)
display(Markdown('**Probe B: stray pixels along the sampling path (mean over 4 configs × 5 frames × 5 samples)**'))
display(pd.DataFrame(rows).set_index('Sampler').T.round(2))

fig, ax = plt.subplots(1, 2, figsize=(14, 4.6))
for s, col, lbl in SAMPLERS:
    r = B[s]['r']; ok = np.isfinite(r)
    ax[0].plot(r[ok], np.nanmean(np.stack(B[s]['wrong']), 0)[ok], color=col, lw=2, label=lbl)
    ax[1].plot(r[ok], 100 * np.nanmean(np.stack(B[s]['persist']), 0)[ok], color=col, lw=2, label=lbl)
for a in ax:
    a.set_xscale('log'); a.invert_xaxis(); a.grid(alpha=0.3); a.set_xlabel('σ/α at that step (sampling runs left → right)')
ax[0].set_yscale('log'); ax[0].set_title('stray pixels (>800 K wrong, off-edge) in the current guess x̂₀', fontsize=10)
ax[1].set_title('% of those pixels still wrong in the final sample', fontsize=10); ax[0].legend(fontsize=9)
fig.suptitle('Probe B: where stray pixels appear and whether they get corrected', fontsize=11)
plt.tight_layout(); plt.show()

In [ ]:
# Filmstrip: the clean-field guess x̂₀ as noise is removed (same initial noise for all three samplers)
z = PROBE[f'{FIG_CONFIG}__probeB']
targets = [30, 5, 1.5, 0.5, 0.15, 0.03]
fig, ax = plt.subplots(3, len(targets) + 2, figsize=(2.5 * (len(targets) + 2), 7.8))
for row, (s, _, lbl) in enumerate(SAMPLERS):
    r = z[f'r_{s}']; T = z[f'traj_{s}'][FRAME, :, 0]
    _ax_img(ax[row, 0], z['gts'][FRAME], 'Ground Truth', fs=8)
    for j, tg in enumerate(targets, 1):
        i = int(np.argmin(np.abs(np.log(np.where(np.isfinite(r), r, 1e9)) - np.log(tg))))
        _ax_img(ax[row, j], T[i], f'x̂₀ at σ/α = {r[i]:.2f}', fs=8)
    _ax_img(ax[row, -1], z[f'final_{s}'][FRAME, 0], 'final sample', fs=8)
    ax[row, 0].set_ylabel(lbl.split(' (')[0], fontsize=9)
fig.suptitle(f'{CFG_LABEL[FIG_CONFIG]}, QSS frame {FRAME + 1}, sample 1: the network\'s clean-field guess during sampling (left → right)', fontsize=11)
plt.tight_layout(); plt.show()

**Reading probe B**

1. DiffusionSR's final stray pixels are **seeded at high noise** (median σ/α ≈ 12), exactly where probe A shows its
   clean-field guess breaking down (see the speckled x̂₀ at σ/α ≈ 30 in the filmstrip).
2. **DDIM locks them in progressively**: a pixel that is wrong at σ/α ≈ 10 has ~20% chance of surviving to the end, one that
   is wrong at σ/α ≈ 0.3 almost 100%.
3. **DDPM's fresh noise corrects them**: only ~1% of the pixels wrong at σ/α ≈ 10 survive.
4. **Flow-Matching rarely makes the mistake in the first place**, but when it does, the mistake persists just as in DDIM,
   because Euler is also deterministic. Its advantage is avoiding errors, not correcting them.

The straight noise-to-data paths of Flow-Matching are *not* the explanation: they only reduce the number of steps needed,
and section 2 showed the step count does not matter here.

## 6. Takeaways

* **Sampler:** sample DiffusionSR/LDM with ancestral DDPM (now the default). DDIM's speckle is a sampling artefact that
  more DDIM steps do not fix.
* **Encoder baseline:** the RRDB encoder alone already reaches ~28.7 dB / MP-MAE ~0.4 px; the diffusion stage mainly adds
  realistic sharpness. Report it as a baseline next to every diffusion result.
* **LDM:** the VAE is adequate (~29.5 dB reconstruction); the 0.07 M-parameter latent U-Net is the bottleneck.
* **DiffusionSR vs Flow-Matching:** the gap in stray pixels comes from the noise-prediction parameterisation, which makes
  DiffusionSR's high-noise clean-field guesses poor. Options if this matters (each needs retraining): predict the clean
  field x₀ directly, or the diffusion "v" target v = α·ε − σ·x₀ (x̂₀ = α·x_t − σ·v̂, errors multiplied by σ ≤ 1), or
  reweight the loss toward the clean field at high noise (e.g. min-SNR weighting).
* **Caveat:** 5 QSS frames and one trained run per configuration; differences below ~0.5 dB between models are within noise.